# Tracing the Spectrum

A point source on a long-slit frame is a narrow ridge along the dispersion axis, a few pixels wide
across the slit. Everything that follows (measuring the sky, collapsing the image to one
dimension, deciding which pixels belong to the object) needs to know where that ridge is, column by
column. That path is the **trace**. `specreduce.tracing` offers three `Trace` classes for finding
it, and the choice between them is a choice about how much to assume:

| Class | Assumption | Use when |
|---|---|---|
| `FlatTrace` | the trace is a horizontal line | the instrument is well aligned, or you need a quick look |
| `FitTrace` | the trace is a smooth low-order curve | almost always; this is the workhorse |
| `ArrayTrace` | nothing; you supply the positions | the shape defeats a polynomial, or the positions come from elsewhere |

This chapter works through all three on the trimmed science frame of
[Chapter 2](02-spectral-images.ipynb), and shows how to tell whether a trace is good enough.

In [ ]:
import sys

sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models
from astropy.visualization import simple_norm
from specreduce.tracing import ArrayTrace, FitTrace, FlatTrace

from common import read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
ny, nx = obj.data.shape
rows = np.arange(ny)

## Finding the source

Before fitting anything, we look at the frame, on the logarithmic stretch of Chapter 1.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2), constrained_layout=True)
ax.imshow(
    obj.data,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    norm=simple_norm(obj.data, stretch="log", vmin=0, vmax=250_000),
)
ax.set(xlabel="Dispersion [pix]", ylabel="Cross-dispersion [pix]");

The bright ridge is TrES-3, and the fainter one above it a second star in the slit. At this scale
both look straight, the impression a trace has to correct. Collapsing the frame along the
dispersion axis gives a rough cross-dispersion profile: rough because it smears a curved trace over
several rows, in which a point source is a narrow peak on the plateau of the sky.

In [ ]:
profile = obj.data.mean(axis=1)
sky_level = np.median(profile)

fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
ax.plot(rows, profile, lw=0.8)
ax.axhline(sky_level, c="0.6", lw=0.8, ls="--")
ax.text(
    365,
    sky_level * 1.15,
    f"sky $\\approx$ {sky_level:.0f} e$^-$",
    ha="right",
    size=8,
    c="0.4",
)
ax.annotate(
    "target\nrow 135",
    (135, profile[135]),
    (147, profile[135] * 0.45),
    size=8,
    va="center",
    arrowprops=dict(arrowstyle="-", lw=0.6, color="0.4"),
)

ax.set(
    yscale="log",
    xlim=(0, ny - 1),
    xlabel="Cross-dispersion [pix]",
    ylabel="Mean signal [e$^-$]",
);

Four sources stand out. The peak at **row 135** is TrES-3, about 60 times the sky level. The other
three are stars that fell in the 40-arcsecond slit: one near row 311 at about 3 percent of the
target, one blended into the wing of TrES-3 near row 117, and a fainter one near row 36. We ignore
all three here.

The peak is narrow, with a FWHM of about **3 pixels**, the seeing disk at this binning, and that
sets the scale for what "accurate enough" means: a trace error of a few tenths of a pixel is
negligible against it, and one of a few pixels is catastrophic. The question is whether the source
stays at row 135. Three cross-dispersion cuts, near the left edge, the middle and the right edge of
the detector:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)

for x0 in (32, 512, 960):
    cut = obj.data[:, x0 - 16 : x0 + 16].mean(axis=1)
    ax.plot(
        rows, cut / cut[np.abs(rows - 135) <= 10].max(), lw=0.9, label=f"column {x0}"
    )

ax.axvline(135, c="0.6", lw=0.8, ls="--")
ax.legend(frameon=False)
ax.set(xlim=(122, 148), xlabel="Cross-dispersion [pix]", ylabel="Normalized signal");

The peak moves by a few pixels between the three cuts, as much as the width of the profile, and a
fixed row cannot follow it.

## `FlatTrace`: assuming a straight line

The simplest possible trace is a constant. `FlatTrace` takes the image and a single
cross-dispersion position, and produces a trace that sits at that row for every column.

In [ ]:
flat_trace = FlatTrace(obj, trace_pos=135)

print(type(flat_trace.trace), flat_trace.trace.shape)
print(flat_trace.trace[:5])

Every `Trace` exposes the same thing, a `.trace` array with one cross-dispersion position per
dispersion column, which is why `Background` and the extraction classes accept any of the three
classes. Overplotted on the frame with only 40 rows shown, which stretches the vertical axis about
25-fold, a flat trace at row 135 visibly misses the ridge.

In [ ]:
def show_trace(ax, traces, ylim=(115, 155), title=None):
    """Display the science frame zoomed on the source, with traces overplotted."""
    ax.imshow(
        obj.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(obj.data, stretch="log", vmin=0, vmax=250_000),
    )
    for tr, label, style in traces:
        ax.plot(np.asarray(tr), label=label, **{"lw": 1.0, **style})
    ax.set(
        xlim=(0, nx - 1),
        ylim=ylim,
        xlabel="Dispersion [pix]",
        ylabel="Cross-dispersion [pix]",
    )
    if title:
        ax.set_title(title, size=9)
    if any(label for _, label, _ in traces):
        ax.legend(frameon=False, loc="upper left", fontsize=8)


fig, ax = plt.subplots(figsize=(8, 2.6), constrained_layout=True)
show_trace(ax, [(flat_trace.trace, "FlatTrace(135)", dict(c="C3", ls="--"))]);

The ridge starts below the line, rises to meet it near the middle of the detector, and falls away
again. `FlatTrace` is not wrong here so much as insufficient, and it has its uses. On a
well-aligned spectrograph it is faster, has no fit to go wrong, and cannot be pulled off by a
cosmic ray, and it is the natural reference for arc frames, whose lines fill the slit and leave no
ridge to follow; Chapter 6 uses it that way.

## `FitTrace`: fitting a curve

`FitTrace` works in three steps:

1. **Bin** the image into groups of columns along the dispersion axis, trading resolution along
   the trace for signal-to-noise in each measurement.
2. **Find the peak** in each bin's cross-dispersion profile, with `max`, `centroid` or `gaussian`.
3. **Fit a 1D model** through the peak positions and evaluate it at every column.

The default model is a `Polynomial1D` of degree 1, a straight line free to tilt but not to bend,
and the cuts above showed the peak rising and then falling, which no line can follow. The question
is how much more freedom to give it.

### Choosing the polynomial degree

With no ground truth, the test is **convergence**: fit a range of degrees and measure how much the
trace changes when one more term is added. While the model misses real structure, each extra term
moves the trace by a substantial fraction of a pixel; once it has enough freedom, the change drops
to the level of the per-bin measurement error. The statistic below is the rms change over all
columns between degree $d$ and $d+1$. The model is `Chebyshev1D` rather than `Polynomial1D`: both
describe the same curves, but a power series in $x$ over 0–1000 is badly conditioned at high
degree, and `astropy` warns about it.

In [ ]:
degrees = range(1, 8)
fitted = {}
for degree in degrees:
    trace = FitTrace(
        obj, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(degree)
    )
    fitted[degree] = np.asarray(trace.trace)

change = {d: np.std(fitted[d + 1] - fitted[d]) for d in degrees[:-1]}

for d, c in change.items():
    print(
        f"degree {d} -> {d + 1}:  span {np.ptp(fitted[d]):.2f} pix,  "
        f"rms change {c:.3f} pix"
    )

Going from degree 1 to 2 moves the trace by 0.78 pixels rms, about a quarter of the profile's
FWHM: the trace is curved, and the quadratic term does most of the work. Degrees 3 and 4 add about
0.05 pixels each, and from degree 5 on the changes are a few hundredths of a pixel or less. We take
degree 4, following the rule of taking the **lowest degree at which an added term stops changing
the trace**, since every extra parameter is another chance to chase noise or to swing past the
ends of the data.

In [ ]:
trace = FitTrace(
    obj, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
)
trace_y = np.asarray(trace.trace)

fig, ax = plt.subplots(figsize=(8, 3.0), constrained_layout=True)
show_trace(
    ax,
    [
        (trace_y, "FitTrace, degree 4", dict(c="C1")),
        (flat_trace.trace, "FlatTrace(135)", dict(c="C3", ls="--", lw=0.8)),
    ],
)

The fitted trace follows the ridge along the whole detector. It starts 1.7 pixels below the flat
line, touches it near the middle, and ends 3.6 pixels below it, a total drift of **3.7 pixels**,
more than the width of the profile.

### Choosing `peak_method`

`FitTrace` has three ways to locate the peak in each bin.

- `max`, the default, takes the brightest pixel. It has no fit to fail and ignores the profile
  shape, which makes it robust on faint, blended or irregular sources, but each measurement is
  quantized to a whole pixel, and only the fit across many bins averages that away; with few bins
  or a high-degree model the rounding is fitted instead.
- `gaussian` fits a Gaussian on a constant baseline to each bin and takes the fitted center. On a
  bright, well-formed profile it is the most precise, and it is what the fits above used. It is
  also the most fragile: on weak or non-Gaussian profiles the per-bin fits become unstable, and a
  fit whose center leaves the frame silently reuses the previous bin's answer.
- `centroid` takes the position that splits the bin's flux in half, over the whole
  cross-dispersion range it is given. Without a `window`, the sky and every other object on the
  slit pull it toward the middle of the slit, most strongly where the target is faintest, and the
  trace leaves the source without warning; with a `window` narrow enough for the target to
  dominate, it agrees closely with `gaussian`. Do not use it without one.

Use `gaussian` on a bright, well-formed profile, and `max` when the profile is too faint or
irregular to fit.

### Confining the fit

`guess` tells `FitTrace` where to start instead of taking the brightest row of the collapsed
frame, and `window` confines the peak search in every bin to `guess ± window` rows. Together they
make the fit deterministic, and they are worth setting even on a bright, isolated target. `window`
cuts both ways: it keeps the fit off a brighter neighbor, but a source that drifts further than
`window` hits the edge of its search region and flattens out, so set it wider than the expected
drift and narrower than the distance to the nearest neighbor.

`disp_bounds=(lo, hi)` confines the fit along the dispersion axis. The bins cover that column range
only, and the model is evaluated at every column, so outside the bounds the trace is an
extrapolation. It is the tool for a source that fades along the trace, where the faint-end bins
would otherwise steer the polynomial. The extrapolation is sound for a smooth optical distortion
and wrong for a kink or a flexure jump, so draw it over the image before trusting it.

On a faint or crowded target these arguments work together: `guess` and `window` select the
source, `disp_bounds` drops the poorly measured columns, and fewer `bins` with a lower-degree model
put more signal into each measurement. On a source only a few times the sky level, expect the peak
methods to disagree, and prefer `max`.

## `ArrayTrace`: supplying positions directly

Sometimes no polynomial will do: the trace has a kink, the instrument flexed mid-exposure, or the
positions come from elsewhere. `ArrayTrace` takes one position per dispersion column verbatim. Its
realistic use on a dataset like this one is a time series: fit the trace once on a bright exposure
and apply it to every frame through `ArrayTrace`, so that all of them are extracted along the same
path. Below are a round trip and what `ArrayTrace` does with an array of the wrong length:

In [ ]:
array_trace = ArrayTrace(obj, trace=trace_y)
print(f"round trip exact: {np.allclose(array_trace.trace, trace_y)}")

short_trace = ArrayTrace(obj, trace=trace_y[:900])
n_masked = np.ma.getmaskarray(short_trace.trace).sum()
print(
    f"900 positions in -> {short_trace.trace.shape[0]} out, {n_masked} of them masked"
)

An array longer than the image is truncated, a shorter one is padded with masked values, and
non-finite entries are masked, so a length bug produces a part-masked trace rather than an
exception; check `np.ma.getmaskarray(trace.trace).sum()` if a trace behaves oddly. With no smooth
model to enforce, whatever noise is in the input goes straight into the trace and the extracted
spectrum, so smooth the positions before handing them over.

## Shifting a trace

Adding or subtracting a number returns a **shifted copy** of a trace, displaced along the
cross-dispersion axis, with the original untouched.

In [ ]:
above = trace + 20
below = trace - 20

print(f"original  {np.asarray(trace.trace)[512]:.2f}")
print(f"+20       {np.asarray(above.trace)[512]:.2f}")
print(f"-20       {np.asarray(below.trace)[512]:.2f}")

fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
show_trace(
    ax,
    [
        (trace_y, "trace", dict(c="C1")),
        (above.trace, "trace + 20", dict(c="C0", ls="--", lw=0.8)),
        (below.trace, "trace - 20", dict(c="C0", ls="--", lw=0.8)),
    ],
    ylim=(100, 170),
);

This is how Chapter 7 places its sky windows: in bands parallel to the trace, offset either side
of the source, so that every pixel in a band stays at the same distance from the object as the
trace curves. Positions that fall off the image are masked rather than raising an error.

## Summary

- A trace is one cross-dispersion position per dispersion column, exposed as `.trace` on all three
  classes, and downstream operations consume it the same way whatever produced it.
- On this frame the source drifts **3.7 pixels** across the detector, more than its **3-pixel**
  FWHM, which no straight trace can follow.
- Validate by convergence: scan the degree, stop at the lowest one where an added term no longer
  moves the trace (degree 4 here), and draw the result over the image.
- `peak_method` depends on the source. `gaussian` is the most precise on a bright, well-formed
  profile and the most fragile elsewhere, `max` is robust but quantized to whole pixels, and
  `centroid` wanders off the source without a `window`.
- `guess`, `window`, `disp_bounds` and `bins` confine the fit. On a faint or crowded source they
  select the right source, keep neighbors and poorly measured columns out, and put more signal
  into each measurement.
- `trace ± delta` gives shifted copies, which is how background windows are placed.

Next: [Arc Lamps](05-arc-lamps.ipynb), the wavelength-reference frames, and the first place the
frame's tilt becomes impossible to ignore.